# Clase 2 - APIs de exchanges centralizados (CEX)

Primero usamos endpoints **publicos** (no requieren credenciales) y despues vemos como se **firman** los requests privados.

> **Nunca subas claves a un repositorio.** Las API keys y secrets se leen de variables de entorno
> (`BINANCE_API_KEY`, `BINANCE_API_SECRET`). Las celdas que necesitan claves se saltean solas si no estan definidas.

In [1]:
import requests
import json
import os

## Datos publicos: libro de ordenes y precio promedio (Binance.us)

In [2]:
binance_api_url = "https://api.binance.us"

uri = "/api/v3/depth"

response = requests.get(binance_api_url + uri, {
    "symbol": "ETHUSDC",
    "limit": 5
})

if response.status_code == 200:
    data = response.json()
    print(data)
else:
    print("[INFO] respuesta inesperada:", response.status_code)

{'lastUpdateId': 382799239, 'bids': [['2362.43000000', '0.42320000'], ['2362.42000000', '0.01160000'], ['2362.25000000', '7.10340000'], ['2362.19000000', '0.74240000'], ['2361.49000000', '0.09510000']], 'asks': [['2362.90000000', '0.42320000'], ['2362.92000000', '2.51250000'], ['2362.93000000', '0.01160000'], ['2363.66000000', '0.09490000'], ['2363.82000000', '0.09480000']]}


In [3]:
uri = "/api/v3/avgPrice"

response = requests.get(binance_api_url + uri, {
    "symbol": "ETHUSDC",
})

if response.status_code == 200:
    data = response.json()
    print(data)
else:
    print("[INFO] respuesta inesperada:", response.status_code)

{'mins': 5, 'price': '2364.67000000', 'closeTime': 1792895877745}


## Requests firmados (HMAC-SHA256)

Los endpoints privados exigen firmar la query con el secret. Abajo se calcula la firma con un secret **falso**
(`dummy-secret-for-demo`) solo para mostrar el mecanismo: este request **no se envia**.

In [4]:
import requests
import time
import hmac
import hashlib


# Credenciales de DEMO, claramente falsas: no sirven para nada
DEMO_API_KEY = "dummy-api-key-for-demo"
DEMO_SECRET = "dummy-secret-for-demo"

# Set up the request parameters
params = {
    'symbol':       'LTCBTC',
    'side':         'BUY',
    'type':         'LIMIT',
    'timeInForce':  'GTC',
    'quantity':     '1',
    'price':        '0.1',
    'recvWindow':   '5000',
    'timestamp':    '1499827319559'
}

# Timestamp the request, for non example requests
# timestamp = int(time.time() * 1000) # UNIX timestamp in milliseconds
# params['timestamp'] = timestamp

# Sign the request
payload = '&'.join([f'{param}={value}' for param, value in params.items()])
signature = hmac.new(
    bytes(DEMO_SECRET, 'latin-1'),
    msg=bytes(payload, 'latin-1'),
    digestmod=hashlib.sha256
).hexdigest()

print(signature)


params['signature'] = signature

# Send the request (NO se envia: con credenciales reales usar la celda siguiente)
headers = {
    'X-MBX-APIKEY': DEMO_API_KEY,
}

#response = requests.post(
#    'https://api.binance.com/api/v3/order',
#    headers=headers,
#    data=params,
#)
#print(response.json())

9b7fb1c8e4b2d1fc7d2a82496339fffec0127a089cf43620a89b678fd68d75e0


### Endpoint privado con claves reales (opcional)
Lee las claves del entorno; sin ellas la celda se saltea.

In [5]:
API_KEY = os.environ.get("BINANCE_API_KEY")
API_SECRET = os.environ.get("BINANCE_API_SECRET")

if not (API_KEY and API_SECRET):
    print("[INFO] set BINANCE_API_KEY/BINANCE_API_SECRET to run this cell")
else:
    q = f"timestamp={int(time.time() * 1000)}&recvWindow=5000"
    sig = hmac.new(API_SECRET.encode(), q.encode(), hashlib.sha256).hexdigest()
    r = requests.get(
        "https://api.binance.us/api/v3/account",
        params=f"{q}&signature={sig}",
        headers={"X-MBX-APIKEY": API_KEY},
    )
    if r.status_code == 200:
        balances = [b for b in r.json()["balances"] if float(b["free"]) > 0]
        print(balances)
    else:
        print("[INFO] status", r.status_code)

[INFO] set BINANCE_API_KEY/BINANCE_API_SECRET to run this cell


## ccxt: una interfaz unica para muchos exchanges

In [6]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("ccxt") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ccxt"])

In [7]:
import ccxt

binance_exchange = ccxt.binanceus()

In [8]:
binance_exchange.fetch_ticker("ETH/USDC")

{'symbol': 'ETH/USDC',
 'timestamp': 1792902115744,
 'datetime': '2026-10-25T04:21:55.744Z',
 'high': 2376.01,
 'low': 2281.22,
 'bid': 2362.43,
 'bidVolume': 0.4232,
 'ask': 2362.9,
 'askVolume': 0.4232,
 'vwap': 2344.11984039,
 'open': 2376.01,
 'close': 2364.67,
 'last': 2364.67,
 'previousClose': 2373.09,
 'change': -11.34,
 'percentage': -0.477,
 'average': 2370.34,
 'baseVolume': 13.4642,
 'quoteVolume': 31561.698355,
 'markPrice': None,
 'indexPrice': None,
 'info': {'symbol': 'ETHUSDC',
  'priceChange': '-11.34000000',
  'priceChangePercent': '-0.477',
  'weightedAvgPrice': '2344.11984039',
  'prevClosePrice': '2373.09000000',
  'lastPrice': '2364.67000000',
  'lastQty': '0.01160000',
  'bidPrice': '2362.43000000',
  'bidQty': '0.42320000',
  'askPrice': '2362.90000000',
  'askQty': '0.42320000',
  'openPrice': '2376.01000000',
  'highPrice': '2376.01000000',
  'lowPrice': '2281.22000000',
  'volume': '13.46420000',
  'quoteVolume': '31561.69835500',
  'openTime': 1792815715744

In [9]:
binance_exchange.fetch_order_book("ETH/USDC", 5)

{'symbol': 'ETH/USDC',
 'bids': [[2362.43, 0.4232],
  [2362.42, 0.0116],
  [2362.19, 0.7424],
  [2361.5, 6.3864],
  [2361.49, 0.0951]],
 'asks': [[2362.9, 0.4232],
  [2362.92, 2.5125],
  [2362.93, 0.0116],
  [2363.66, 0.0949],
  [2363.82, 0.0948]],
 'timestamp': None,
 'datetime': None,
 'nonce': 382799241}

In [10]:
# Velas OHLCV (timestamp, open, high, low, close, volume)
binance_exchange.fetch_ohlcv("ETH/USDC", "1h", limit=5)

[[1792886400000, 2359.29, 2359.29, 2359.29, 2359.29, 0.0116],
 [1792890000000, 2354.25, 2354.25, 2354.25, 2354.25, 0.0254],
 [1792893600000, 2364.67, 2364.67, 2364.67, 2364.67, 0.0116],
 [1792897200000, 2364.67, 2364.67, 2364.67, 2364.67, 0.0],
 [1792900800000, 2364.67, 2364.67, 2364.67, 2364.67, 0.0]]

In [11]:
# Datos privados con ccxt (saldos): requieren claves en el entorno
if not (API_KEY and API_SECRET):
    print("[INFO] set BINANCE_API_KEY/BINANCE_API_SECRET to run this cell")
else:
    private_exchange = ccxt.binanceus({"apiKey": API_KEY, "secret": API_SECRET})
    balance = private_exchange.fetch_balance()
    print({k: v for k, v in balance["total"].items() if v})

[INFO] set BINANCE_API_KEY/BINANCE_API_SECRET to run this cell


## Siguiente: DEX en Polygon

La comparacion de precios SushiSwap vs QuickSwap (contratos on-chain con web3.py) esta en `clase_2.py`:
`python clase_2.py --blocks 3 --seconds 30`. Usa RPCs publicos sin API key; para uno propio definir `RPC_URL_POLYGON` / `WSS_URL_POLYGON`.